![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 13:  Claims Data Analysis



**Health Informatics in Python** · Part IV: Advanced Topics · Module 13 of 16

---



Clinical informatics records *care*. **Claims data** records *payment* — the
structured bills providers send payers after a visit, stay, or fill. This module
treats claims as an informatics data type: what they contain, how they are coded
and classified, how to clean them, and how to roll them up into **member-level**
cost, risk, and quality measures.


## Learning objectives

By the end of this module you will be able to:

1. Describe a **claim** (who, what, why, where, how much) and how it differs from an EHR.
2. Map claim codes to **classification systems** (ICD-10, CPT, HCC, DRG).
3. **Clean** raw claims: duplicates, adjudication status, financial integrity.
4. Build **member-level features**: PMPM, HCC risk score, Charlson index, utilization.
5. **Risk-stratify** a population and compute a simple **care-gap** (HEDIS-like) measure.


## Dataset

A synthetic **Medicare Advantage–style book**: an **eligibility** file (members)
and a **claims** file (service-level rows with ICD-10, CPT, place of service, and
billed / allowed / paid amounts). We inject duplicate claims and a few dirty
values so the cleaning step has something to do. No real patients or payers.


In [ ]:
# --- Self-contained synthetic claims generator ---
import numpy as np
import pandas as pd

def make_synthetic_claims(n_members=400, n_claims=2400, seed=42):
    """Eligibility file + service-level claims for a fake MA-style plan."""
    rng = np.random.default_rng(seed)

    # 1. Member eligibility (who is covered, and for how many months)
    members = pd.DataFrame({
        "member_id": [f"M{1000+i}" for i in range(n_members)],
        "age": rng.integers(45, 90, size=n_members),
        "sex": rng.choice(["male", "female"], size=n_members, p=[0.47, 0.53]),
        "region": rng.choice(["Northeast", "Southeast", "Midwest", "West"], size=n_members),
        "plan_type": rng.choice(["HMO", "PPO", "PFFS"], size=n_members, p=[0.50, 0.35, 0.15]),
        "enroll_months": rng.integers(6, 13, size=n_members),
    })
    # Chronic flags with roughly realistic prevalence (used to bias claim codes)
    members["has_diabetes"]     = rng.binomial(1, 0.22, n_members)
    members["has_chf"]          = rng.binomial(1, 0.10, n_members)
    members["has_copd"]         = rng.binomial(1, 0.12, n_members)
    members["has_ckd"]          = rng.binomial(1, 0.14, n_members)
    members["has_depression"]   = rng.binomial(1, 0.16, n_members)
    members["has_hypertension"] = rng.binomial(1, 0.45, n_members)
    members["has_cancer"]       = rng.binomial(1, 0.06, n_members)

    # Inject a few dirty eligibility values (~4%)
    bad = rng.choice(n_members, size=max(8, n_members // 25), replace=False)
    members.loc[bad[: len(bad)//2], "age"] = np.nan
    members.loc[bad[len(bad)//2 :], "sex"] = "U"

    # 2. Code dictionaries: diagnosis (why) and procedure (what)
    icd = {
        "E11.9": "Type 2 diabetes without complications",
        "I50.32": "Chronic systolic heart failure",
        "J44.9": "COPD, unspecified",
        "N18.3": "CKD stage 3",
        "F32.9": "Major depressive disorder, unspecified",
        "I10": "Essential hypertension",
        "Z12.11": "Encounter for screening for malignant neoplasm of colon",
        "M54.5": "Low back pain",
        "J06.9": "Acute upper respiratory infection",
        "Z00.00": "Encounter for general adult medical examination",
    }
    cpt = {
        "99213": ("Office visit, established", 150),
        "99214": ("Office visit, moderate complexity", 250),
        "99283": ("ED visit, moderate", 1200),
        "99285": ("ED visit, high severity", 2500),
        "99232": ("Subsequent hospital care", 180),
        "93306": ("Echocardiogram", 800),
        "71046": ("Chest X-ray, 2 views", 300),
        "82947": ("Glucose, quantitative", 50),
        "36415": ("Venipuncture", 25),
        "27447": ("Total knee arthroplasty", 15000),
    }
    pos = {"11": "Office", "21": "Inpatient hospital",
           "23": "Emergency room", "22": "Outpatient hospital", "81": "Laboratory"}

    # Condition → likely primary ICD (so coded claims track the member flags)
    cond_icd = {
        "has_diabetes": "E11.9", "has_chf": "I50.32", "has_copd": "J44.9",
        "has_ckd": "N18.3", "has_depression": "F32.9",
        "has_hypertension": "I10", "has_cancer": "Z12.11",
    }
    icd_keys, cpt_keys, pos_keys = list(icd), list(cpt), list(pos)

    def pick_icd(mid):
        row = members.loc[members["member_id"] == mid].iloc[0]
        coded = [code for flag, code in cond_icd.items() if row[flag] == 1]
        if coded and rng.random() < 0.65:
            return rng.choice(coded)
        return rng.choice(icd_keys)

    # 3. Service-level claims
    member_ids = rng.choice(members["member_id"], size=n_claims)
    cpt_drawn = rng.choice(cpt_keys, size=n_claims)
    billed = np.array([cpt[c][1] for c in cpt_drawn]) * rng.uniform(0.85, 1.40, n_claims)
    allowed = billed * rng.uniform(0.55, 0.80, n_claims)
    paid = allowed * rng.uniform(0.70, 0.95, n_claims)
    start = pd.Timestamp("2023-01-01")
    svc = [start + pd.Timedelta(days=int(d)) for d in rng.integers(0, 364, n_claims)]

    claims = pd.DataFrame({
        "claim_id": [f"CLM{i+1:06d}" for i in range(n_claims)],
        "member_id": member_ids,
        "service_date": svc,
        "icd10_primary": [pick_icd(m) for m in member_ids],
        "cpt_code": cpt_drawn,
        "place_of_service": rng.choice(pos_keys, size=n_claims, p=[0.45, 0.12, 0.12, 0.18, 0.13]),
        "billed_amount": billed.round(2),
        "allowed_amount": allowed.round(2),
        "paid_amount": paid.round(2),
        "provider_npi": rng.integers(1000000000, 1999999999, size=n_claims),
        "claim_status": rng.choice(["Paid", "Denied", "Pending"], size=n_claims, p=[0.88, 0.08, 0.04]),
    })

    # ~72% of diabetic members get at least one glucose test so the HEDIS-like gap is realistic
    for mid in members.loc[members["has_diabetes"] == 1, "member_id"]:
        if rng.random() < 0.72:
            hit = claims.index[claims["member_id"] == mid]
            if len(hit):
                claims.loc[rng.choice(hit), "cpt_code"] = "82947"

    # Inject ~3% duplicate submissions (same member / date / CPT / billed)
    dup_idx = rng.choice(n_claims, size=int(0.03 * n_claims), replace=False)
    claims = pd.concat([claims, claims.iloc[dup_idx]], ignore_index=True)

    return {
        "members": members,
        "claims": claims,
        "icd": icd,
        "cpt": cpt,
        "pos": pos,
    }

book = make_synthetic_claims()
members, claims = book["members"], book["claims"]
print("Tables:", f"members ({len(members)}), claims ({len(claims)} including duplicates)")
print("Date range:", claims["service_date"].min().date(), "→", claims["service_date"].max().date())
print("Claim statuses:", claims["claim_status"].value_counts().to_dict())


## 13.1 What claim data is (and is not)

A **claim** is the structured record a provider submits to a payer (commercial
insurer, Medicare, Medicaid) to request **reimbursement for services already
delivered**.

```
Patient receives care → provider submits a claim → payer adjudicates → claim is recorded
```

| Layer | What it captures |
|---|---|
| **Patient / member** | ID, age, sex, geography, eligibility months |
| **Provider** | NPI, specialty, facility |
| **Clinical (coded)** | ICD-10 *why*, CPT/HCPCS *what*, revenue codes |
| **Financial** | Billed, allowed, paid, member cost-share |
| **Context** | Dates, place of service, claim status (paid / denied / pending) |

**Claim types.** Professional (CMS-1500, office visits), institutional (UB-04,
hospital stays), pharmacy (NCPDP), dental.

**Why informatics cares.** Claims are how ICD and CPT *travel* through the
payment system (Module 3). They are also the main administrative feed into
warehouses (Module 14), population programs (Module 12), and risk adjustment.

**Limits.** Claims are **not** the medical record. They lack vitals, labs,
notes, and often secondary diagnoses. They are billing-driven, lagged (days to
months), and sensitive to how providers code. Use them for *what was billed and
paid*, not as a substitute for clinical truth.


## 13.2 Anatomy of a claim line

Each row below is one **service line**: a coded reason (ICD), a coded service
(CPT), a setting (place of service), and three dollar amounts. Payers typically
keep **paid** claims for analytics; denied and pending rows stay in operational
queues.


In [ ]:
# Peek at a few raw claim lines and decode CPT / POS for readability
sample = claims.head(8).copy()
sample["cpt_desc"] = sample["cpt_code"].map(lambda c: book["cpt"][c][0])
sample["pos_desc"] = sample["place_of_service"].map(book["pos"])
cols = ["claim_id", "member_id", "service_date", "icd10_primary",
        "cpt_code", "cpt_desc", "pos_desc", "billed_amount",
        "allowed_amount", "paid_amount", "claim_status"]
print(sample[cols].to_string(index=False))


## 13.3 Classification tools on claims

Claims do not store "this patient is high risk" as a field. Analysts **group**
raw codes into payment and risk categories. Module 3 introduced the codes; here
they become *groupers*.

| Tool | Setting | Job on a claim |
|---|---|---|
| **ICD-10-CM** | All | Diagnosis / reason for visit |
| **CPT / HCPCS** | Outpatient / professional | Procedure / service billed |
| **MS-DRG** | Inpatient | Bundled hospital payment per stay |
| **HCC** | Medicare Advantage | Risk-adjust capitation from diagnoses |
| **CCI / LACE** | All | Comorbidity / readmission risk from codes |

**HCC (Hierarchical Condition Categories)** collapse related ICD codes into
categories that predict next-year cost. A higher HCC score means a sicker
member and, under Medicare Advantage, a higher payment to the plan. The weights
below are **educational approximations**, not the CMS model.


In [ ]:
# Educational ICD-10 → HCC crosswalk (NOT the official CMS-HCC model)
hcc_map = pd.DataFrame({
    "icd10": ["E11.9", "I10", "J44.9", "N18.3", "F32.9", "I50.32", "Z12.11", "J06.9"],
    "description": [
        "Type 2 diabetes without complications",
        "Essential hypertension",
        "COPD, unspecified",
        "CKD stage 3",
        "Major depression, unspecified",
        "Chronic systolic HF",
        "Colon cancer screening (no HCC)",
        "Acute URI (acute — usually no HCC)",
    ],
    "hcc": ["HCC19", "HCC85", "HCC111", "HCC137", "HCC59", "HCC85", None, None],
    "weight": [0.318, 0.302, 0.335, 0.289, 0.309, 0.368, None, None],
})
print("ICD-10 codes on this book's claims, with a toy HCC mapping")
print(hcc_map.to_string(index=False))


## 13.4 Cleaning and validating claims

Production claims are messy: **duplicates** (resubmits), **reversals**, mixed
**adjudication status**, missing demographics, and amounts that violate
`paid ≤ allowed ≤ billed`. Analytics almost always start from **paid** claims
after those checks — the same data-quality instinct as Module 8, applied to
administrative data.


In [ ]:
raw_n = len(claims)

# 1. Drop duplicate submissions (same member, date, CPT, billed amount)
claims_clean = claims.drop_duplicates(
    subset=["member_id", "service_date", "cpt_code", "billed_amount"]
).copy()
print(f"Duplicates removed: {raw_n - len(claims_clean):,}")

# 2. Keep paid claims for cost / utilization analytics
claims_clean = claims_clean[claims_clean["claim_status"] == "Paid"].copy()
print(f"After status filter (Paid only): {len(claims_clean):,}")

# 3. Repair eligibility: numeric age, recode unknown sex
members_clean = members.copy()
members_clean["age"] = pd.to_numeric(members_clean["age"], errors="coerce")
members_clean["age"] = members_clean["age"].fillna(members_clean["age"].median()).clip(0, 110).astype(int)
members_clean["sex"] = members_clean["sex"].replace("U", "female")

# 4. Financial integrity: paid cannot exceed allowed
over = claims_clean["paid_amount"] > claims_clean["allowed_amount"]
claims_clean.loc[over, "paid_amount"] = claims_clean.loc[over, "allowed_amount"]
print(f"Paid>allowed rows capped: {int(over.sum()):,}")

claims_clean = claims_clean[claims_clean["paid_amount"] >= 0]
print(f"Clean claims: {len(claims_clean):,} | clean members: {len(members_clean):,}")


## 13.5 Member-level features: PMPM, HCC, utilization

A claim is a **transaction**. Care management and risk models need a **member**.
We aggregate paid claims to one row per member:

| Feature | Meaning |
|---|---|
| **PMPM** | Paid amount ÷ enrollment months — the core actuarial unit |
| **HCC score** | Age/sex baseline + weights for coded chronic conditions |
| **CCI** | Weighted comorbidity count (Charlson-style) |
| **Utilization** | ER, inpatient, and office visits from place of service |

Normalizing HCC so the book average is **1.0** is how plans talk about "sicker
than average" (score > 1) versus "healthier" (score < 1).


In [ ]:
# Join paid claims to eligibility, then roll up to the member
df = claims_clean.merge(members_clean, on="member_id", how="left")

member_feat = df.groupby("member_id").agg(
    total_paid=("paid_amount", "sum"),
    n_claims=("claim_id", "count"),
    n_icd=("icd10_primary", "nunique"),
    n_providers=("provider_npi", "nunique"),
    er_visits=("place_of_service", lambda s: (s == "23").sum()),
    ip_visits=("place_of_service", lambda s: (s == "21").sum()),
    office_visits=("place_of_service", lambda s: (s == "11").sum()),
).reset_index()

member_feat = member_feat.merge(members_clean, on="member_id", how="left")
member_feat["enroll_months"] = member_feat["enroll_months"].clip(lower=1)
member_feat["pmpm"] = (member_feat["total_paid"] / member_feat["enroll_months"]).round(2)

# Toy HCC: age/sex factor + condition weights (educational, not CMS)
hcc_w = {
    "has_diabetes": 0.318, "has_chf": 0.368, "has_copd": 0.335,
    "has_ckd": 0.289, "has_depression": 0.309, "has_hypertension": 0.118,
    "has_cancer": 0.450,
}

def age_sex_factor(row):
    base = 0.28 + row["age"] * 0.007
    if row["sex"] == "female" and row["age"] >= 65:
        base += 0.04
    return round(base, 3)

member_feat["age_sex"] = member_feat.apply(age_sex_factor, axis=1)
member_feat["hcc_raw"] = (
    member_feat["age_sex"] + sum(member_feat[c] * w for c, w in hcc_w.items())
).round(3)
member_feat["hcc"] = (member_feat["hcc_raw"] / member_feat["hcc_raw"].mean()).round(3)

# Simplified Charlson weights on the same flags
cci_w = {"has_diabetes": 1, "has_chf": 1, "has_copd": 1, "has_ckd": 2, "has_cancer": 2}
member_feat["cci"] = sum(member_feat[c] * w for c, w in cci_w.items())
cond_cols = list(hcc_w)
member_feat["n_chronic"] = member_feat[cond_cols].sum(axis=1)

print(f"Members with claims: {len(member_feat):,}")
print(f"Mean PMPM: ${member_feat['pmpm'].mean():,.2f}")
print(f"Mean HCC (normalized): {member_feat['hcc'].mean():.3f}  (1.0 = book average)")
print(f"Mean CCI: {member_feat['cci'].mean():.2f}")
print(f"Total ER visits: {member_feat['er_visits'].sum():,}")
print(member_feat[["member_id", "age", "pmpm", "hcc", "cci", "n_chronic"]].head().to_string(index=False))


### Milestone 1 — risk stratification

Care programs do not treat every member the same. A simple **rule engine**
combines HCC, inpatient use, and ER use into four tiers — the same pattern as
the CDS rule engine in Module 11, now at **population** scale.

| Tier | Rule of thumb | Typical action |
|---|---|---|
| 4 Critical | HCC ≥ 1.8 or IP ≥ 3 or ER ≥ 4 | Intensive case management |
| 3 High | HCC ≥ 1.2 or IP ≥ 1 or ER ≥ 2 | Care-manager outreach |
| 2 Rising | HCC ≥ 0.9 or 2+ chronic conditions | Prevention / coaching |
| 1 Low | Everyone else | Wellness |


In [ ]:
def assign_tier(row):
    hcc, er, ip = row["hcc"], row["er_visits"], row["ip_visits"]
    if hcc >= 1.8 or ip >= 3 or er >= 4:
        return "4 - Critical"
    if hcc >= 1.2 or ip >= 1 or er >= 2:
        return "3 - High"
    if hcc >= 0.9 or row["n_chronic"] >= 2:
        return "2 - Rising"
    return "1 - Low"

member_feat["risk_tier"] = member_feat.apply(assign_tier, axis=1)

# Disease registries from the same flags (claims-derived cohorts)
member_feat["registry_diabetes"] = member_feat["has_diabetes"] == 1
member_feat["registry_chf"] = member_feat["has_chf"] == 1
member_feat["registry_copd"] = member_feat["has_copd"] == 1

tier = (member_feat.groupby("risk_tier")
        .agg(members=("member_id", "count"),
             total_paid=("total_paid", "sum"),
             mean_pmpm=("pmpm", "mean"))
        .reset_index())
tier["cost_share_pct"] = (tier["total_paid"] / tier["total_paid"].sum() * 100).round(1)
print("Risk-tier distribution")
print(tier.to_string(index=False))
print("\nRegistries: diabetes", int(member_feat["registry_diabetes"].sum()),
      "| CHF", int(member_feat["registry_chf"].sum()),
      "| COPD", int(member_feat["registry_copd"].sum()))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

order = ["1 - Low", "2 - Rising", "3 - High", "4 - Critical"]
colors = ["#55A868", "#4C72B0", "#DD8452", "#C44E52"]
counts = member_feat["risk_tier"].value_counts().reindex(order).fillna(0)
axes[0].barh(order, counts.values, color=colors)
axes[0].set_xlabel("members")
axes[0].set_title("Members by risk tier")

paid = member_feat.groupby("risk_tier")["total_paid"].sum().reindex(order).fillna(0)
axes[1].pie(paid, labels=order, colors=colors, autopct="%1.0f%%",
            textprops={"fontsize": 8})
axes[1].set_title("Share of paid amount")

plt.tight_layout()
plt.show()


## 13.6 Quality measures and care gaps (HEDIS-like)

**HEDIS**-style measures ask: of the members who *should* have received a
service, what share *did*? Numerators and denominators are built from claims
(and sometimes pharmacy or labs). Members in the denominator but not the
numerator are a **care gap** — an outreach list.

Here the denominator is members with diabetes on file; the numerator is those
with at least one **glucose test (CPT 82947)** in the year — a simplified
stand-in for HbA1c testing, not a certified HEDIS engine.


In [ ]:
# Denominator: members flagged with diabetes
diabetics = set(member_feat.loc[member_feat["has_diabetes"] == 1, "member_id"])

# Numerator: at least one glucose test billed in the year
tested = set(
    claims_clean.loc[claims_clean["cpt_code"] == "82947", "member_id"]
)
tested_diabetics = diabetics & tested

rate = (len(tested_diabetics) / len(diabetics)) if diabetics else float("nan")
benchmark = 0.80  # illustrative national-style target, not a real HEDIS rate
gap = rate - benchmark

print(f"Diabetes denominator: {len(diabetics)}")
print(f"With ≥1 glucose test (CPT 82947): {len(tested_diabetics)}")
print(f"Testing rate: {rate:.1%}  |  benchmark: {benchmark:.0%}  |  gap: {gap:+.1%}")

care_gap = member_feat[
    member_feat["member_id"].isin(diabetics - tested)
][["member_id", "age", "pmpm", "hcc", "risk_tier"]].sort_values("pmpm", ascending=False)

print(f"\nCare gap: {len(care_gap)} diabetic members with no glucose test on file")
print("Highest-PMPM gaps (outreach priority):")
print(care_gap.head(8).to_string(index=False))


In [ ]:
# Visual: plan rate vs an illustrative benchmark
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar([0], [rate * 100], width=0.45, color="#4C72B0", label="plan")
ax.bar([0.5], [benchmark * 100], width=0.45, color="#8C8C8C", label="benchmark")
ax.axhline(benchmark * 100, color="#C44E52", ls="--", lw=1)
ax.set_xticks([0, 0.5])
ax.set_xticklabels(["Plan (claims)", "Benchmark"])
ax.set_ylabel("percent of diabetics tested")
ax.set_title("HEDIS-like: glucose testing among members with diabetes")
ax.set_ylim(0, 100)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 13.7 Claims next to the EHR

Claims and EHR data answer different questions. Informatics pipelines often
**join** them (eligibility + claims + clinical) inside a warehouse — the subject
of the next module.

| | Claims | EHR |
|---|---|---|
| **Unit** | Billable service | Clinical encounter / observation |
| **Strength** | Complete for billed care; cost; population coverage | Vitals, labs, notes, orders |
| **Weakness** | Billing-driven, lagged, thin clinically | Incomplete across sites; not built for payment |
| **Typical use** | PMPM, HCC, HEDIS, fraud, network | CDS, quality of *documented* care, phenotyping |

The data-scientist workflow on a claims book is: **acquire → clean → feature
(PMPM, HCC, CCI) → classify / risk-stratify → quality gaps → action**. This
module ran the middle of that pipeline on synthetic data. Production work adds
837/835 EDI, pharmacy claims, official groupers, and actuarial sign-off.


## Exercises

1. Recompute **PMPM using allowed amount** instead of paid. How much does the
   book average move, and why would a plan care?
2. Build an **age-banded** table of mean HCC and mean PMPM (a standard actuarial
   cut). Does cost rise smoothly with age in this synthetic book?
3. Define a second care gap — e.g. members with hypertension (`I10`) and **no
   office visit** (POS `11`) — and list the highest-PMPM members in the gap.



## Key takeaways

- **Claims** are the financial/administrative backbone of US healthcare, not a
  substitute for the EHR.
- **ICD, CPT, HCC, DRG** are how a bill becomes a comparable, groupable record.
- Analytics run on **cleaned paid claims** rolled up to the **member-month**.
- **PMPM + HCC + utilization** feed risk tiers, registries, and HEDIS-like gaps.



---
*Next: Module 14 - Clinical Data Warehousing and Analytics-Ready Data.*
